# Dev Baseline Analysis (Pipeline v1)

This notebook analyzes the full dev-split baseline run of the combined mapper + verifier pipeline.

**Input:** `results/dev_baseline_meaningful.csv` and `results/dev_baseline_all.csv`

**Source script:** `src/pipeline/run_dev_baseline.py` (all dev sentences from `hi_hdtb-ud-dev.conllu`)

Analysis only. No mapper, verifier, or rule changes.

**Decision labels:**
- `confirmed` / `ambiguous`: verifier-backed
- `mapping_hypothesis`: unverified mapper guess (UD label only)
- `no_decision`: no usable Karaka candidate

## 1. Load CSV Files

In [1]:
import csv
from collections import Counter
from pathlib import Path

MEANINGFUL_PATH = Path("../results/dev_baseline_meaningful.csv")
ALL_PATH = Path("../results/dev_baseline_all.csv")


def load_csv(filepath):
    with open(filepath, encoding="utf-8", newline="") as f:
        return list(csv.DictReader(f))


meaningful = load_csv(MEANINGFUL_PATH)
all_rows = load_csv(ALL_PATH)

total_tokens = len(all_rows)
print(f"Total tokens (all):        {total_tokens}")
print(f"Meaningful rows:           {len(meaningful)}")
print(f"Sentences (unique sent_id): {len(set(r['sent_id'] for r in all_rows))}")

Total tokens (all):        35217
Meaningful rows:           7019
Sentences (unique sent_id): 1659


## 2. Summary Counts

In [2]:
final_counts_all = Counter(row["final_decision"] for row in all_rows)
final_counts_meaningful = Counter(row["final_decision"] for row in meaningful)
mapper_status_counts = Counter(row["mapper_status"] for row in all_rows)
rule_counts = Counter(row["verifier_rule_id"] for row in all_rows if row["verifier_rule_id"])
deprel_counts = Counter(row["deprel"] for row in all_rows)

print("final_decision (all tokens):")
for decision, count in sorted(final_counts_all.items()):
    pct = 100 * count / total_tokens
    print(f"  {decision:<22} {count:>6}  ({pct:.2f}%)")
print()

print("final_decision (meaningful only):")
for decision, count in sorted(final_counts_meaningful.items()):
    pct = 100 * count / len(meaningful)
    print(f"  {decision:<22} {count:>6}  ({pct:.2f}%)")
print()

print("mapper_status (all tokens):")
for status, count in sorted(mapper_status_counts.items()):
    pct = 100 * count / total_tokens
    print(f"  {status:<22} {count:>6}  ({pct:.2f}%)")
print()

print("verifier_rule_id (tokens where a rule fired):")
rule_total = sum(rule_counts.values())
for rule_id, count in sorted(rule_counts.items()):
    pct_all = 100 * count / total_tokens
    pct_rules = 100 * count / rule_total
    print(f"  {rule_id:<6} {count:>6}  ({pct_all:.2f}% of all, {pct_rules:.1f}% of rule hits)")
print()

print("Top deprels (all tokens):")
for deprel, count in deprel_counts.most_common(15):
    pct = 100 * count / total_tokens
    print(f"  {deprel:<14} {count:>6}  ({pct:.2f}%)")

final_decision (all tokens):
  ambiguous                 840  (2.39%)
  confirmed                1715  (4.87%)
  mapping_hypothesis       4464  (12.68%)
  no_decision             28198  (80.07%)

final_decision (meaningful only):
  ambiguous                 840  (11.97%)
  confirmed                1715  (24.43%)
  mapping_hypothesis       4464  (63.60%)

mapper_status (all tokens):
  context_dependent        3036  (8.62%)
  evidence_only            6674  (18.95%)
  mapped                   3983  (11.31%)
  no_karaka                1659  (4.71%)
  unsupported             19865  (56.41%)

verifier_rule_id (tokens where a rule fired):
  R1        556  (1.58% of all, 21.8% of rule hits)
  R2        847  (2.41% of all, 33.2% of rule hits)
  R3        312  (0.89% of all, 12.2% of rule hits)
  R4        341  (0.97% of all, 13.3% of rule hits)
  R5        499  (1.42% of all, 19.5% of rule hits)

Top deprels (all tokens):
  case             6674  (18.95%)
  compound         4038  (11.47%)
  nmo

## 3. Example Rows by final_decision

In [3]:
DISPLAY_COLS = [
    "sent_id",
    "token_form",
    "deprel",
    "case_marker",
    "final_decision",
    "final_candidates",
    "verifier_rule_id",
    "mapper_status",
]


def show_examples(example_rows, title, max_examples=5):
    print(title)
    print("=" * 70)
    if not example_rows:
        print("(no rows)")
        print()
        return
    for i, row in enumerate(example_rows[:max_examples], start=1):
        print(f"Example {i} ({row['sent_id']}, {row['token_form']})")
        print(f"  Sentence: {row['sentence_text']}")
        for col in DISPLAY_COLS:
            print(f"  {col}: {row[col]}")
        print()


confirmed_rows = [r for r in meaningful if r["final_decision"] == "confirmed"]
ambiguous_rows = [r for r in meaningful if r["final_decision"] == "ambiguous"]
hypothesis_rows = [r for r in meaningful if r["final_decision"] == "mapping_hypothesis"]

show_examples(confirmed_rows, f"Confirmed ({len(confirmed_rows)} total)")
show_examples(ambiguous_rows, f"Ambiguous ({len(ambiguous_rows)} total)")
show_examples(hypothesis_rows, f"Mapping hypothesis ({len(hypothesis_rows)} total)")

Confirmed (1715 total)
Example 1 (dev-s1, काल)
  Sentence: रामायण काल में भगवान राम के पुत्र कुश की राजधानी कुशावती को 483 ईसा पूर्व बुद्ध ने अपने अंतिम विश्राम के लिए चुना ।
  sent_id: dev-s1
  token_form: काल
  deprel: obl
  case_marker: में
  final_decision: confirmed
  final_candidates: Adhikaraṇa
  verifier_rule_id: R2
  mapper_status: context_dependent

Example 2 (dev-s1, बुद्ध)
  Sentence: रामायण काल में भगवान राम के पुत्र कुश की राजधानी कुशावती को 483 ईसा पूर्व बुद्ध ने अपने अंतिम विश्राम के लिए चुना ।
  sent_id: dev-s1
  token_form: बुद्ध
  deprel: nsubj
  case_marker: ने
  final_decision: confirmed
  final_candidates: Kartā
  verifier_rule_id: R1
  mapper_status: mapped

Example 3 (dev-s2, प्राचीनकाल)
  Sentence: मल्‍लों की राजधानी होने के कारण प्राचीनकाल में इस स्‍थान का अत्‍यंत महत्‍व था ।
  sent_id: dev-s2
  token_form: प्राचीनकाल
  deprel: obl
  case_marker: में
  final_decision: confirmed
  final_candidates: Adhikaraṇa
  verifier_rule_id: R2
  mapper_status: context_depe

## 4. Summary

Full baseline report: `docs/verifier_v1_dev_baseline.md`

Regenerate dev CSVs:

```bash
python src/pipeline/run_dev_baseline.py
```